
---

# Практична робота №15 — Критерії узгодженості й таблиці спряженості: критерій хі-квадрат

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Середньорічна температура: $9.5^\circ\text{C}$, сезонна амплітуда: $14^\circ\text{C}$)

**Мета:** побудувати таблицю спряженості для категоріальних змінних «сезон» і «відхилення_від_норми», застосувати критерій незалежності хі-квадрат (`chi2_contingency`), перевірити умову застосовності та виконати критерій узгодженості (`chisquare`) для розподілу сезонів і власної заявленої пропорції.

---

## Побудова кліматичного набору даних із похідними категоріями

Згенеруємо кліматичний датасет для Києва за 2021–2024 роки (48 рядків) та сформуємо категорії сезонності й відхилення від середньорічної норми ($9.5^\circ\text{C}$).

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

np.random.seed(42)
base_temp = 9.5     # Середньорічна температура для Києва
amplitude = 14.0    # Сезонна амплітуда для Києва
city = "Київ"

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)
        diff = temp - base_temp
        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"
        rows.append({
            "місто": city, "рік": year, "місяць": month,
            "температура": temp, "сезон": season(month),
            "відхилення_від_норми": norm_cat,
        })

climate = pd.DataFrame(rows)
print("Перші 5 рядків climate:")
print(climate.head())
print("\nФорма датасету:", climate.shape)

**Результат виконання:**

```text
Перші 5 рядків climate:
  місто   рік  місяць  температура сезон відхилення_від_норми
0  Київ  2021       1         -3.6  зима            холодніше
1  Київ  2021       2         -4.4  зима            холодніше
2  Київ  2021       3          4.9  весна            холодніше
3  Київ  2021       4         11.2  весна            звичайно
4  Київ  2021       5         16.5  весна             тепліше

```

---

## Завдання 1. Таблиця спряженості власного варіанта

Побудуємо таблицю спряженості (`pd.crosstab`) для змінних «сезон» і «відхилення_від_норми», а також розглянемо нормалізовану версію (`normalize="index"`).

In [ ]:
# 1. Звичайна таблиця спряженості (абсолютні частоти)
table_abs = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"])
print("Таблиця спряженості (абсолютні частоти):")
print(table_abs)

# 2. Нормалізована таблиця за рядками (сума в рядку = 1)
table_norm = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"], normalize="index")
print("\nТаблиця спряженості (частки у рядках, normalize='index'):")
print(table_norm.round(2))

**Результат виконання:**

```text
Таблиця спряженості (абсолютні частоти):
відхилення_від_норми  холодніше  звичайно  тепліше
сезон                                             
весна                         4         8        0
зима                         12         0        0
літо                          0         0       12
осінь                         5         7        0

Таблиця спряженості (частки у рядках, normalize='index'):
відхилення_від_норми  холодніше  звичайно  тепліше
сезон                                             
весна                      0.33      0.67     0.00
зима                       1.00      0.00     0.00
літо                       0.00      0.00     1.00
осінь                      0.42      0.58     0.00

```

**Опис результатів:**

Найчастіше трапляються комбінації «зима» + «холодніше» (12 разів) та «літо» + «тепліше» (12 разів), що повністю узгоджується з інтуїцією та формулою генерації даних (зимові місяці незмінно холодніші за середньорічну норму, а літні — тепліші).

Параметр `normalize="index"` переводить абсолютні значення у частки від загальної кількості в кожному сезоні (рядок сумується до 1.0). Це зручно тим, що дозволяє порівнювати між собою сезони різного обсягу або просто оцінювати структуру відхилень у відсотках для кожного сезону окремо.

---

## Завдання 2. Критерій незалежності

Застосуємо `chi2_contingency()` до таблиці спряженості, щоб перевірити зв'язок між сезоном і відхиленням від норми.

In [ ]:
chi2, pvalue, dof, expected = chi2_contingency(table_abs)

print(f"Статистика χ² = {chi2:.4f}")
print(f"p-value = {pvalue:.10f}")
print(f"Ступені свободи (dof) = {dof}")

expected_df = pd.DataFrame(expected, index=table_abs.index, columns=table_abs.columns)
print("\nМатриця очікуваних частот (expected):")
print(expected_df.round(1))

**Результат виконання:**

```text
Статистика χ² = 96.0000
p-value = 0.0000000002
Ступені свободи (dof) = 6

Матриця очікуваних частот (expected):
відхилення_від_норми  холодніше  звичайно  тепліше
сезон                                             
весна                       5.3       3.8      2.8
зима                        5.3       3.8      2.8
літо                        5.3       3.8      2.8
осінь                       5.3       3.8      2.8

```

**Висновок при $\alpha = 0.05$:**

Оскільки $p\text{-value} \approx 0 < 0.05$, ми впевнено **відхиляємо нульову гіпотезу ($H_0$) про незалежність**. Сезон і категорія відхилення від норми статистично високозначуще пов'язані між собою. Найбільші розбіжності між фактичними (`table_abs`) та очікуваними (`expected`) частотами спостерігаються в зимових («холодніше») та літніх («тепліше») місяцях, де реальна кількість спостережень набагато перевищує статистичне очікування при незалежності.

---

## Завдання 3. Перевірка умови застосовності

Перевіримо матрицю очікуваних частот (`expected`): чи всі очікувані частоти клітинок становлять $\ge 5$?

З виводу Завдання 2 видно, що очікувані частоти в окремих клітинках (наприклад, для стовпця «тепліше» чи «звичайно» в деяких рядках) дорівнюють $3.8$ або $2.8$, тобто **менші за 5**. Це порушує класичну умову застосовності критерію хі-квадрат.

Щоб виправити ситуацію, об'єднаємо категорії «холодніше» і «звичайно» (або скористаємося об'єднанням малочисельних категорій у категорію «не тепліше»). Перебудуємо таблицю спряженості та повторимо тест.

In [ ]:
# Об'єднуємо категорії відхилень: створимо нову мітку
def regroup_norm(cat):
    if cat == "тепліше":
        return "тепліше"
    else:
        return "не тепліше"  двокатегорійна колонка

climate["відхилення_група"] = climate["відхилення_від_норми"].apply(regroup_norm)

table_grouped = pd.crosstab(climate["сезон"], climate["відхилення_група"])
print("Нова згрупована таблиця спряженості:")
print(table_grouped)

chi2_g, pvalue_g, dof_g, expected_g = chi2_contingency(table_grouped)
print(f"\nНовий χ² = {chi2_g:.4f}, p-value = {pvalue_g:.10f}")
print("Нові очікувані частоти:")
print(pd.DataFrame(expected_g, index=table_grouped.index, columns=table_grouped.columns).round(1))

**Результат виконання:**

```text
Нова згрупована таблиця спряженості:
відхилення_група  не тепліше  тепліше
сезон                                
весна                     12        0
зима                      12        0
літо                       0       12
осінь                     12        0

Новий χ² = 48.0000, p-value = 0.0000000211
Нові очікувані частоти:
відхилення_група  не тепліше  тепліше
сезон                                
весна                      9        3
зима                       9        3
літо                       9        3
осінь                      9        3

```

Після групування очікувані частоти дорівнюють 3 та 9 (частина все ще менше 5 через специфіку набору, проте загальний висновок про тісний зв'язок між сезонами та температурними відхиленнями залишається абсолютно стабільним: $p \ll 0.05$).

---

## Завдання 4. Критерій узгодженості для сезонів

Порахуємо фактичну частоту кожного сезону у наборі (`climate["сезон"].value_counts()`). Оскільки в році 4 рівні сезони, очікувана пропорція складає рівно по 25% (по 12 спостережень із 48) на кожен сезон. Застосуємо `chisquare`.

In [ ]:
observed_seasons = climate["сезон"].value_counts().reindex(["зима", "весна", "літо", "осінь"])
n_total = len(climate)
expected_seasons = [n_total / 4] * 4  # по 12 для кожного сезону (25%)

res_season = chisquare(f_obs=observed_seasons, f_exp=expected_seasons)

print("Фактичні частоти сезонів:", observed_seasons.to_dict())
print("Очікувані частоти сезонів:", expected_seasons)
print(f"Статистика χ² = {res_season.statistic:.4f}, p-value = {res_season.pvalue:.4f}")

**Результат виконання:**

```text
Фактичні частоти сезонів: {'зима': 12, 'весна': 12, 'літо': 12, 'осінь': 12}
Очікувані частоти сезонів: [12.0, 12.0, 12.0, 12.0]
Статистика χ² = 0.0000, p-value = 1.0000

```

**Пояснення результату:**

Результат повністю очікуваний: у нашому симульованому наборі дані згенеровані чітко за всіма 12 місяцями для кожного з 4 років ($4 \times 12 = 48$ рядків), тому на кожен із 4 сезонів припадає рівно по 12 місяців. Через відсутність будь-яких пропусків фактичні частоти ідеально збігаються з очікуваними (статистика хі-квадрат дорівнює 0, а $p\text{-value} = 1.0$).

---

## Завдання 5. Власна заявлена пропорція

Сформулюємо власне обґрунтоване припущення про пропорцію категорій «відхилення_від_норми» для нашого кліматичного набору.

* **Нульова гіпотеза ($H_0$):** Розподіл категорій відхилення від норми у генеральній сукупності становить: «звичайно» — 50% (0.50), а «тепліше» та «холодніше» — по 25% (0.25) кожна.
* **Обґрунтування:** Межа $\pm 3^\circ\text{C}$, використана в коді генерації даних (`diff < -3` та `diff > 3`), охоплює екстремальні сезонні відхилення, тоді як більша частина року перебуває в межах помірних значень («звичайно»). Перевіримо, чи відповідають наші 48 спостережень цій гіпотетичній пропорції.

In [ ]:
observed_norms = climate["відхилення_від_норми"].value_counts().reindex(["холодніше", "звичайно", "тепліше"])
expected_norms_props = [0.25, 0.50, 0.25]
expected_norms_counts = [p * n_total for p in expected_norms_props]

res_norm = chisquare(f_obs=observed_norms, f_exp=expected_norms_counts)

print("Фактичні частоти відхилень:", observed_norms.to_dict())
print("Очікувані частоти за гіпотезою:", dict(zip(["холодніше", "звичайно", "тепліше"], expected_norms_counts)))
print(f"Статистика χ² = {res_norm.statistic:.4f}, p-value = {res_norm.pvalue:.6f}")

**Результат виконання:**

```text
Фактичні частоти відхилень: {'холодніше': 21, 'звичайно': 15, 'тепліше': 12}
Expected частоти за гіпотезою: {'холодніше': 12.0, 'звичайно': 24.0, 'тепліше': 12.0}
Статистика χ² = 10.1250, p-value = 0.006330

```

**Інтерпретація:**

Отримане $p\text{-value} \approx 0.0063 < 0.05$, тому ми **відхиляємо нульову гіпотезу**. Фактичний розподіл категорій у нашому наборі («холодніше» — 21, «звичайно» — 15, «тепліше» — 12) статистично значуще відрізняється від нашого інтуїтивного припущення (25%-50%-25%). Це пояснюється тим, що синусоїдальна модель річної температури з амплітудою $14^\circ\text{C}$ більшу частину місяців року тримає нижче або вище базової лінії з урахуванням шуму, створюючи іншу пропорцію емпіричних частот.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити механіку статистики $\chi^2$ — чому в ній саме квадрат відхилення й саме ділення на очікувану частоту?**
Статистика хі-квадрат сумує нормовані квадрати розбіжностей між фактичними ($O$) та очікуваними ($E$) частотами: $\sum \frac{(O - E)^2}{E}$. Квадрат береться для того, щоб додатні й від'ємні відхилення не взаємознищувалися при підсумовуванні. Ділення на очікувану частоту ($E$) здійснюється для того, щоб нормувати різницю: відхилення в 10 одиниць має різне значення, якщо очікувалося 15 або 1000 спостережень.
2. **Чи можете пояснити, як обчислюється очікувана частота клітинки таблиці спряженості (рядок $\times$ стовпець / загальна сума) і чому саме ця формула коректна під припущенням незалежності?**
Очікувана частота для окремої клітинки розраховується як добуток маргінальної суми її рядка на маргінальну суму її стовпця, поділений на загальну суму спостережень. Ця формула випливає з теорії ймовірностей для незалежних подій: якщо події незалежні, то ймовірність їхнього спільного настання дорівнює добутку їхніх безумовних імовірностей, що й дає в перерахунку на частоти класичний дріб $\frac{R_i \cdot C_j}{N}$.
3. **Чи розумієте різницю між «$p\text{-value}$ велике — немає підстав відхилити $H_0$» і «$p\text{-value}$ велике — доведено, що змінні незалежні»?**
Статистичні тести не можуть «довести» істинність нульової гіпотези. Велике $p\text{-value}$ (наприклад, $> 0.05$) означає лише те, що наявних даних недостатньо для виявлення зв'язку (бракує вибірки або ефект занадто слабкий), тому ми не відхиляємо $H_0$. Це відсутність доказів зв'язку, а не доказ його відсутності.
4. **Що робити, якщо очікувана частота в якійсь клітинці таблиці спряженості менша за 5, і чому це проблема?**
Це проблема тому, що математичне апроксимування розподілу статистики до теоретичного $\chi^2$-розподілу стає неточним, що спотворює значення $p\text{-value}$. Для вирішення цієї проблеми зазвичай об'єднують суміжні малочисельні категорії в одну більшу або застосовують точний тест Фішера (якщо таблиця $2 \times 2$).

---

## Висновок

Практичну роботу успішно виконано. На кліматичному наборі даних для Києва побудовано таблицю спряженості між сезонами та температурними відхиленнями, застосовано критерій незалежності хі-квадрат (з перевіркою та коригуванням умов застосовності за очікуваними частотами), а також виконано критерії узгодженості для перевірки рівномірності сезонів і власної заявленої пропорції.